<a href="https://colab.research.google.com/github/CPTR295/Sample-LLMs/blob/main/Tokenizer_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from importlib.metadata import version
print("torch version:",version('torch'))
print("tiktoken version",version('tiktoken'))

torch version: 2.11.0+cu128
tiktoken version 0.13.0


In [15]:
import os
import requests
file_path = '/content/the-verdict.txt'
if not os.path.exists(file_path):
  url = 'https://raw.githubusercontent.com/CPTR295/Sample-LLMs/main/the-verdict.txt'
  response = requests.get(url,timeout=30)
  response.raise_for_status()
  with open(file_path,'wb') as file:
    file.write(response.content)


In [16]:
with open(file_path,'r',encoding='utf-8') as f:
  text = f.read()
print("Total number of characters:",len(text))


Total number of characters: 20479


In [17]:
import re
#Split all words and special characters using re framework to get all tokens
#Example
test = "hello, world."
result = re.split(r'([,.:;?_!"()\']|--|\s)', test)
result  = [item.strip() for item in result if item.strip()]
print(result)

['hello', ',', 'world', '.']


In [18]:
#Same for main data
preprocessed = result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [19]:
print(len(preprocessed)) #Total number of tokens

4690


In [20]:
#Convert Tokens to IDs
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [21]:
vocab = {token:interger for interger,token in enumerate(all_words)} #My vocabulory from data

In [22]:
for i,item in enumerate(vocab.items()):
  print(item)
  if i>20:
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)


In [23]:
class SimpleTokenierV1: #Its job is to give token id for every token from data
  def __init__(self,vocab):
    self.str_to_int = vocab
    self.int_to_str = {interger:token for token,interger in vocab.items()}
  def encode(self,text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    return [self.str_to_int[token] for token in preprocessed]

  def decode(self,ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text


In [28]:
tokenizer = SimpleTokenierV1(vocab)
#sample_text = 'Hello world' will not work as hello is not present in data hence we may need to include <unk>
sample_text = 'the last he painted'
ids = tokenizer.encode(sample_text)
print(ids)

[988, 602, 533, 746]


In [29]:
tokenizer.decode(ids)

'the last he painted'

In [30]:
#Add special tokens its can be [BOS],[EOS],[PAD],[UNK] etc
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:interger for interger,token in enumerate(all_tokens)}
len(vocab)

1132

In [31]:
for i,t in enumerate(list(vocab.items())[-5:]):
  print(t)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [33]:
#Update tokenizer to include special tokens
class SimpleTokenierV2:
  def __init__(self,vocab):
    self.str_to_int = vocab
    self.int_to_str = {interger:token for token,interger in vocab.items()}
  def encode(self,text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    preprocessed = [ item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
    ids = [self.str_to_int[token] for token in preprocessed]
    return ids
  def decode(self,ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [36]:
tokenizer = SimpleTokenierV2(vocab)
sample_text = 'Hello the last he painted'
ids = tokenizer.encode(sample_text)
print(ids)

[1131, 988, 602, 533, 746]


In [37]:
tokenizer.decode(ids)

'<|unk|> the last he painted'

In [39]:
#Sample GTP2 - BytePairEncoding
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')
sample_text = 'hello , Hi This is someunkplace <|endoftext|> Welcome!'
ids = tokenizer.encode(sample_text,allowed_special={"<|endoftext|>"})
print(ids)

[31373, 837, 15902, 770, 318, 617, 2954, 5372, 220, 50256, 19134, 0]


In [41]:
string = tokenizer.decode(ids)
print(string)

hello , Hi This is someunkplace <|endoftext|> Welcome!


In [42]:
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [45]:
#Sliding window for a given list of IDs predict the next word
enc_sample = enc_text[50:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [46]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [47]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [49]:
#Implement a dataloader in pytorch
import torch
from torch.utils.data import Dataset,DataLoader

In [50]:
#To extract chunks from datasets
class GPTDatasetV1(Dataset):
  def __init__(self,txt,tokenizer,max_lenght,stride):
    self.input_ids=[]
    self.target_ids=[]
    token_ids = tokenizer.encode(txt,allowed_special={"<|endoftext|>"})
    assert len(token_ids)>max_lenght,"Number  of tokenized inputs must at least be equal to max_length"
    for i in range(0,len(token_ids)-max_lenght,stride):
      input_chunk = token_ids[i:i+max_lenght]
      target_chunk = token_ids[i+1:i+max_lenght+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))
  def __len__(self):
    return len(self.input_ids)
  def __getitem__(self,idx):
    return self.input_ids[idx],self.target_ids[idx]


In [51]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [53]:
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [54]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)
#it works like an iterator
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [55]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [56]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [ ]:
#Now we can create word embedding and position embedding
vocab_size = 6
output_dim = 3
max_length = 6
token_embeddings = torch.nn.Embedding(vocab_size, output_dim)
context_length = max_length
pos_embeddings = torch.nn.Embedding(context_length, output_dim)
input_embeddings = token_embeddings + pos_embeddings
#Only for show